# Cours interactif - K-fold et ensembling dans `train_kfold_ensembling_plantar_model.py`

Objectif : comprendre pourquoi on a ajoute `--cv-folds` dans le script, comment le K-fold fonctionne, et pourquoi l'ensembling peut ameliorer la prediction finale.

A la fin, tu dois pouvoir expliquer cette commande :

```bash
.venv/bin/python scripts/train_kfold_ensembling_plantar_model.py --model mlp --cv-folds 5
```

## 0. Idee generale

Avant, le pipeline utilisait un split unique :

```text
Train -> Validation -> Test
```

Maintenant, on garde toujours un **test final separe**, mais on remplace la validation unique par plusieurs validations :

```text
Train+Validation ---------------> K folds
Test final ---------------------> jamais utilise pendant l'entrainement
```

Le test final reste sacre : il sert seulement a mesurer le score a la fin.

In [ ]:
# A executer seulement si une importation echoue plus bas.
# Dans Jupyter, %pip installe les paquets dans le kernel courant.
# %pip install pandas numpy scikit-learn joblib matplotlib

In [ ]:
import sys
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, GroupKFold

try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
if not (ROOT / 'scripts').exists():
    raise RuntimeError('Lance ce notebook depuis la racine du repo ou depuis notebooks/.')

sys.path.insert(0, str(ROOT / 'scripts'))
import train_kfold_ensembling_plantar_model as tpm

print('Racine du projet :', ROOT)
print('Dossier Events present :', (ROOT / 'Events').exists())
print('Dossier Plantar_activity_reference present :', (ROOT / 'Plantar_activity_reference').exists())

## 1. Pourquoi le split unique peut etre fragile

Avec un seul split train/validation/test, le score de validation depend beaucoup du hasard du decoupage.

Exemple : si certaines actions faciles tombent surtout dans la validation, le score semble trop bon. Si des actions difficiles tombent surtout dans la validation, le score semble trop mauvais.

Le K-fold reduit cet effet : chaque exemple du bloc `train+validation` devient validation exactement une fois.

In [ ]:
labels = np.array([
    'walk'] * 12 +
    ['jump'] * 12 +
    ['squat'] * 12
)

folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
rows = []
for fold, (train_idx, val_idx) in enumerate(folds.split(np.zeros(len(labels)), labels), start=1):
    rows.append({
        'fold': fold,
        'train_size': len(train_idx),
        'val_size': len(val_idx),
        'val_distribution': dict(pd.Series(labels[val_idx]).value_counts().sort_index()),
    })

pd.DataFrame(rows)

## 2. StratifiedKFold

`StratifiedKFold` essaie de garder la meme proportion de classes dans chaque fold.

C'est utile ici car on a beaucoup d'actions differentes. Sans stratification, un fold pourrait manquer une action ou contenir une distribution trop differente.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, (fold, (_, val_idx)) in zip(axes, enumerate(folds.split(np.zeros(len(labels)), labels), start=1)):
    pd.Series(labels[val_idx]).value_counts().sort_index().plot(kind='bar', ax=ax)
    ax.set_title(f'Fold {fold}')
    ax.set_xlabel('Classe')
    ax.set_ylabel('Nb exemples')
plt.tight_layout()

## 3. Le protocole utilise dans notre script

Dans `train_kfold_ensembling_plantar_model.py`, quand `--cv-folds 5` est active :

1. On construit le dataset `X, y` depuis les fichiers `insoles.csv` et `classif.csv`.
2. On separe un test final, par defaut `15%` des exemples.
3. On fusionne l'ancien train et l'ancienne validation en un bloc `train+validation`.
4. On lance le K-fold sur ce bloc.
5. On entraine un modele par fold.
6. On predit le test final avec les 5 modeles.
7. On moyenne les probabilites, puis on garde la classe la plus probable.

In [ ]:
args = SimpleNamespace(
    plantar_root=ROOT / 'Plantar_activity_reference',
    events_root=ROOT / 'Events',
    output=ROOT / 'outputs' / 'notebook_demo.joblib',
    mode='event',
    window_seconds=2.0,
    stride_seconds=1.0,
    min_samples=20,
    max_samples_per_event=12,
    split='random',
    model='extra_trees',
    n_estimators=30,
    n_neighbors=5,
    epochs=4,
    batch_size=64,
    learning_rate=0.001,
    alpha=0.0001,
    hidden_layers='64',
    patience=2,
    overfit_gap=0.10,
    val_drop=0.03,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    max_files=8,
    val_size=0.15,
    test_size=0.15,
    cv_folds=3,
    strict=False,
    no_save=True,
)

data_ready = args.plantar_root.exists() and args.events_root.exists()
if data_ready:
    X, y, metas, feature_names = tpm.build_dataset(args)
    print('X shape :', X.shape)
    print('Nombre de classes :', len(set(y)))
    print('Exemples de labels :', sorted(set(y))[:5])
else:
    print('Donnees absentes : la suite fonctionne en mode lecture, mais les cellules dataset seront ignorees.')

## 4. Garder un test final separe

Point cle : le K-fold ne remplace pas le test final.

Le test final doit rester en dehors de toutes les decisions. Si on utilise le test pour choisir les hyperparametres, il devient une validation deguisee, et le score final est trop optimiste.

In [ ]:
if data_ready:
    split = tpm.split_train_val_test(
        X, y, metas,
        split=args.split,
        val_size=args.val_size,
        test_size=args.test_size,
        random_state=args.random_state,
    )
    X_train, X_val, X_test, y_train, y_val, y_test, train_idx, val_idx, test_idx = split
    train_val_idx = np.concatenate([train_idx, val_idx])

    print('Ancien train :', len(train_idx))
    print('Ancienne validation :', len(val_idx))
    print('Train+validation pour le K-fold :', len(train_val_idx))
    print('Test final conserve a part :', len(test_idx))
else:
    print('Cellule ignoree car les donnees ne sont pas presentes.')

## 5. Construire les folds reels

Le script appelle `cv_splitter(...)`.

- Si `--split random`, il utilise `StratifiedKFold`.
- Si `--split subject`, il utilise `GroupKFold` pour eviter de melanger les sujets entre train et validation.
- Si `--split sequence`, il utilise aussi `GroupKFold`, mais au niveau sequence.

In [ ]:
if data_ready:
    y_train_val = y[train_val_idx]
    folds_real = tpm.cv_splitter(
        y_train_val=y_train_val,
        metas=metas,
        train_val_idx=train_val_idx,
        split=args.split,
        requested_folds=args.cv_folds,
        random_state=args.random_state,
    )

    fold_rows = []
    for fold, (fold_train_local, fold_val_local) in enumerate(folds_real, start=1):
        fold_rows.append({
            'fold': fold,
            'train_samples': len(fold_train_local),
            'val_samples': len(fold_val_local),
            'val_classes': len(set(y_train_val[fold_val_local])),
        })

    display(pd.DataFrame(fold_rows))
else:
    print('Cellule ignoree car les donnees ne sont pas presentes.')

## 6. Pourquoi GroupKFold peut etre plus strict

Quand on fait un split aleatoire, un meme sujet peut se retrouver dans train et validation. Le modele peut alors apprendre des habitudes propres au sujet.

Avec `GroupKFold`, on dit :

```text
Un sujet entier va soit dans train, soit dans validation.
```

C'est plus proche d'une vraie situation ou l'on veut reconnaitre l'activite d'une nouvelle personne.

In [ ]:
subjects = np.array(['S01'] * 4 + ['S02'] * 4 + ['S03'] * 4 + ['S04'] * 4)
y_demo = np.array(['walk', 'jump', 'walk', 'jump'] * 4)

gkf = GroupKFold(n_splits=4)
rows = []
for fold, (tr, va) in enumerate(gkf.split(np.zeros(len(y_demo)), y_demo, groups=subjects), start=1):
    rows.append({
        'fold': fold,
        'train_subjects': sorted(set(subjects[tr])),
        'val_subjects': sorted(set(subjects[va])),
    })

pd.DataFrame(rows)

## 7. Ensembling : l'idee

Chaque fold donne un modele different, car il n'a pas vu exactement les memes exemples pendant son entrainement.

Au lieu de choisir un seul modele, on demande aux 5 modeles leur probabilite pour chaque classe, puis on moyenne :

```text
modele 1 -> [0.70, 0.20, 0.10]
modele 2 -> [0.55, 0.35, 0.10]
modele 3 -> [0.40, 0.50, 0.10]

moyenne -> [0.55, 0.35, 0.10]
classe finale -> classe 0
```

Cela reduit souvent la variance : une erreur isolee d'un modele peut etre compensee par les autres.

In [ ]:
proba_model_1 = np.array([[0.70, 0.20, 0.10]])
proba_model_2 = np.array([[0.55, 0.35, 0.10]])
proba_model_3 = np.array([[0.40, 0.50, 0.10]])

stacked = np.stack([proba_model_1, proba_model_2, proba_model_3], axis=0)
mean_proba = stacked.mean(axis=0)
classes = np.array(['walk', 'jump', 'squat'])

pretty_proba = {str(label): float(round(value, 3)) for label, value in zip(classes, mean_proba[0])}
print('Probabilites moyennes :', pretty_proba)
print('Prediction finale :', classes[mean_proba.argmax(axis=1)][0])

## 8. Pourquoi aligner les classes ?

Dans certains folds, il peut arriver qu'un modele voie moins de classes, surtout sur un petit dataset de test.

Le script utilise `predict_proba_aligned(...)` pour garantir que les colonnes de probabilite sont toujours dans le meme ordre.

Sans cela, on pourrait moyenner des colonnes qui ne correspondent pas a la meme action.

In [ ]:
print('Fonctions ajoutees dans le script :')
for name in ['cv_splitter', 'train_cv_ensemble', 'predict_proba_aligned', 'train_estimator']:
    print('-', name, '->', hasattr(tpm, name))

## 9. Lancer un smoke test depuis le notebook

La cellule suivante lance un entrainement minuscule pour verifier que le pipeline marche.

Elle utilise `extra_trees` et seulement quelques fichiers pour aller vite. Ce n'est pas le score final du projet.

In [ ]:
import subprocess

RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    command = [
        sys.executable,
        str(ROOT / 'scripts' / 'train_kfold_ensembling_plantar_model.py'),
        '--max-files', '8',
        '--model', 'extra_trees',
        '--n-estimators', '30',
        '--cv-folds', '2',
        '--no-save',
    ]
    result = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
else:
    print('Passe RUN_SMOKE_TEST a True pour lancer le smoke test.')

## 10. Commande complete recommandee

Pour reproduire la meilleure configuration observee :

```bash
.venv/bin/python scripts/train_kfold_ensembling_plantar_model.py \
  --model mlp \
  --cv-folds 5 \
  --epochs 80 \
  --batch-size 64 \
  --hidden-layers 256,128 \
  --learning-rate 0.001 \
  --alpha 0.0001 \
  --patience 12 \
  --output outputs/plantar_activity_mlp_cv_ensemble.joblib
```

Resultat observe : environ `0.813` d'accuracy sur le test final.

## 11. Lire les resultats sauvegardes

Le script sauvegarde quatre fichiers :

```text
outputs/plantar_activity_mlp_cv_ensemble.joblib
outputs/plantar_activity_mlp_cv_ensemble_history.csv
outputs/plantar_activity_mlp_cv_ensemble_metrics.json
outputs/plantar_activity_mlp_cv_ensemble_test_predictions.csv
```

Le fichier le plus utile pour comprendre le resultat est souvent `_metrics.json`.

In [ ]:
import json

metrics_path = ROOT / 'outputs' / 'plantar_activity_mlp_cv_ensemble_metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print('Accuracy ensemble :', round(metrics['accuracy'], 3))
    print('Balanced accuracy :', round(metrics['balanced_accuracy'], 3))
    print('Nombre de folds :', metrics['cv']['folds'])
    display(pd.DataFrame(metrics['cv']['fold_summaries']))
else:
    print('Pas encore de metrics complet. Lance la commande complete pour generer ce fichier.')

## 12. Verifier l'overfitting

Avec K-fold, on regarde deux choses :

1. **Train vs validation dans chaque fold** : si train est tres haut et validation stagne, le modele memorise trop.
2. **Validation moyenne vs test final** : si la validation est bonne mais le test final chute beaucoup, on a peut-etre trop optimise les choix sur la validation.

Dans notre run, la validation des folds est autour de `0.76-0.78`, et le test ensemble est autour de `0.813`. C'est coherent, meme si les courbes train montrent encore un risque d'overfitting.

In [ ]:
history_path = ROOT / 'outputs' / 'plantar_activity_mlp_cv_ensemble_history.csv'
if history_path.exists():
    history = pd.read_csv(history_path)
    best_by_fold = history.sort_values('val_accuracy').groupby('fold').tail(1)
    display(best_by_fold[['fold', 'epoch', 'train_accuracy', 'val_accuracy']])

    fig, ax = plt.subplots(figsize=(10, 4))
    for fold, part in history.groupby('fold'):
        ax.plot(part['epoch'], part['val_accuracy'], label=f'fold {int(fold)}')
    ax.set_title('Validation accuracy par fold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation accuracy')
    ax.legend()
    plt.tight_layout()
else:
    print('Historique absent. Lance la commande complete pour tracer les courbes.')

## 13. Checklist a retenir

Avant de croire un score, pose ces questions :

- Le test final est-il reste separe jusqu'a la fin ?
- Les folds sont-ils stratifies si le split est aleatoire ?
- Les sujets sont-ils groupes si on veut tester une vraie generalisation par personne ?
- La normalisation et l'imputation sont-elles apprises seulement sur le train du fold ?
- Les probabilites des modeles sont-elles alignees avant la moyenne ?
- Le score final est-il accompagne d'une balanced accuracy et d'une matrice de confusion ?

## 14. Mini quiz

1. Pourquoi ne doit-on pas utiliser le test final pour choisir `hidden_layers` ou `alpha` ?
2. Quelle difference entre `StratifiedKFold` et `GroupKFold` ?
3. Pourquoi moyenner `predict_proba` est souvent plus stable que choisir le meilleur fold ?
4. Si `train_accuracy = 0.99` et `val_accuracy = 0.70`, que soupconnes-tu ?

Reponses rapides :

1. Sinon le test devient une validation cachee.
2. `StratifiedKFold` equilibre les classes ; `GroupKFold` garde les groupes separes.
3. L'ensemble reduit la variance des modeles individuels.
4. Probable overfitting.